In [25]:
import nltk
from nltk.tokenize import word_tokenize
import emoji
import re
from utils2 import get_dict
import numpy as np


In [26]:
corpus = 'who ❤️ "word embeddings" in 2026? I do !!!'

In [27]:
data = re.sub(r'[,!?;-]','.',corpus) # This line replaces punctuation characters with .
data =nltk.word_tokenize(data) # This splits the text into tokens.
data

['who',
 '❤️',
 '``',
 'word',
 'embeddings',
 "''",
 'in',
 '2026',
 '.',
 'I',
 'do',
 '...']

In [28]:
data =  [ ch.lower() for ch in data
            if ch.isalpha() # Keeps only alphabetic words.
            or ch == "."  # keeps sentence separators.
            or emoji.is_emoji(ch) # keeps emojis.
            ]
data

['who', '❤️', 'word', 'embeddings', 'in', '.', 'i', 'do']

## Cleaning and tokenization 

In [29]:
def tokenize (corpus : str )-> list  : 
    data = re.sub(r'[,!?;-]','.',corpus)
    data =nltk.word_tokenize(data)
    data =  [ ch.lower() for ch in data
            if ch.isalpha() # Keeps only alphabetic words.
            or ch == "."  # keeps sentence separators.
            or emoji.is_emoji(ch) # keeps emojis.
            ]
    return data

In [30]:
corpus = 'who ❤️ "word embeddings" in 2026? I do !!!'
print (tokenize(corpus))

['who', '❤️', 'word', 'embeddings', 'in', '.', 'i', 'do']


## Sliding window of words in python

In [31]:
def get_windows (words, C = 2):
    i = C
    while i < len (words) - C:
        center_word = words[i]
        context_word = words[i-C : i] + words[i+1 : i+C+1 ]
        yield context_word,center_word #using yield instead of return. return immediately exit the function, 
                                       #yield, pauses the function, sends back a value temporarily, and allows the function to continue later from the same point.
        i +=1

In [32]:
corpus = 'I am happy because I am learning'
words = tokenize(corpus)

print (tokenize(corpus))

['i', 'am', 'happy', 'because', 'i', 'am', 'learning']


In [33]:
for x, y in get_windows(words, 2):
    print (f'{x}\t{y}')

['i', 'am', 'because', 'i']	happy
['am', 'happy', 'i', 'am']	because
['happy', 'because', 'am', 'learning']	i


## Transforming words into vectors 

### Mapping words to indices and indices to words

In [34]:
# Get word2Ind and Ind2word dicts for the tokenized corpus
word2Ind, In2word = get_dict (words)

In [35]:
word2Ind

{'am': 0, 'because': 1, 'happy': 2, 'i': 3, 'learning': 4}

In [36]:
# print value for the key '2' within In2word dictionary 
print("Word which has index 2: ",In2word[2])

Word which has index 2:  happy


In [37]:
# Save length of word2Ind dict into the 'V' variable
V = len(word2Ind)

print("Size of vocabulary: ",V) 

Size of vocabulary:  5


### Getting one-hot word vectors


In [38]:
# Save index of word 'happy' into the 'n' Variable
n =word2Ind['happy']
n


2

In [39]:
center_word_vector = np.zeros(V) # vector filled with zeros with the same size as the vocabulary 
center_word_vector


array([0., 0., 0., 0., 0.])

In [40]:
len(center_word_vector) == V # Assert that vocab length and center_word_vector length is the same

True

In [41]:
# Replace element number 'n' with a 1
center_word_vector[n] = 1


In [42]:
center_word_vector

array([0., 0., 1., 0., 0.])

In [43]:
# all in 1 function
def word_to_one_hot_vector (word, word2Ind, V):
    one_hot_vector = np.zeros(V)
    one_hot_vector[word2Ind[word]] = 1
    return one_hot_vector

In [44]:
word_to_one_hot_vector('happy', word2Ind, V)

array([0., 0., 1., 0., 0.])

### Getting  context word vectors


In [47]:
def context_words_to_vector (context_words, word2Ind, V):
    context_words_vectors = [word_to_one_hot_vector(w ,word2Ind,V) for w in context_words ] 
    context_words_vectors = np.mean(context_words_vectors, axis=0)
    return context_words_vectors

In [48]:
context_words = ['i', 'am', 'because', 'i']
print(context_words_to_vector (context_words, word2Ind, V))

[0.25 0.25 0.   0.5  0.  ]
